In [2]:
# ============================================================
# THE DELEGATION BOUNDARY: LLMs as GDD Co-Authors
# CSYE 7270 — Take-Home Midterm Demo Notebook
# ============================================================
# HOW TO RUN:
# 1. pip install openai
# 2. Replace "YOUR_API_KEY_HERE" with your OpenAI API key
# 3. Run cells top to bottom in order
# ============================================================

# -------------------------------------------------------
# CELL 1 — SETUP
# -------------------------------------------------------
# Install dependency if needed:
# !pip install openai

from openai import OpenAI

import os
API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")

def call_gpt(system_prompt, user_prompt, label=""):
    """Single helper to call GPT-4o and print output cleanly."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        max_tokens=800,
        temperature=0.7
    )
    output = response.choices[0].message.content
    print(output)
    return output

print("Setup complete. Ready to run.")


Setup complete. Ready to run.


In [3]:
# -------------------------------------------------------
# CELL 2 — BASELINE SYSTEM PROMPT
# -------------------------------------------------------
#
# DESIGN DECISION (non-obvious):
# We use the same system prompt for both calls so that any
# difference in output is caused by the user prompt alone —
# not by different instructions. This makes the comparison
# scientifically clean and the failure attributable to
# underspecification, not to model configuration.
#
SYSTEM_PROMPT = "You are a game design document author. Be specific and numerically precise."


In [4]:
# -------------------------------------------------------
# CELL 3 — CALL 1: UNDERSPECIFIED PROMPT (The Happy Path)
# -------------------------------------------------------
#
# This is the dangerous call. The prompt describes the
# surface feature (bones as weapons/modifiers) but omits
# the core mechanic (bones are consumable and permanently
# lost on use). The model will fill this gap with genre
# convention: accumulation, shops, predictable pacing.
# That output is what the essay calls "hollow authority."
#
UNDERSPECIFIED_PROMPT = """Write the resource economy section for a roguelike dungeon crawler
where the player collects and equips bones as weapons and stat modifiers.
Include drop rates, upgrade nodes, and a floor-by-floor progression curve."""

output_call1 = call_gpt(SYSTEM_PROMPT, UNDERSPECIFIED_PROMPT, "CALL 1 — UNDERSPECIFIED PROMPT")


  CALL 1 — UNDERSPECIFIED PROMPT
### Resource Economy: Bones as Weapons and Stat Modifiers

#### Overview
In this roguelike dungeon crawler, bones serve as the primary resource for both combat and character progression. Players collect bones from defeated enemies and environmental sources within the dungeon. These bones can be used as weapons or equipped as stat modifiers, enhancing the player's abilities. The resource economy is designed to encourage strategic decision-making and adaptation to the random nature of drops.

#### Bone Types and Drop Rates
1. **Basic Bones:**
   - **Drop Rate:** 50% chance from common enemies.
   - **Function:** Basic attack weapons with no additional stat modifiers.
   - **Examples:** Femur Club, Rib Sword.

2. **Rare Bones:**
   - **Drop Rate:** 20% chance from elite enemies, 5% from common enemies.
   - **Function:** Provide unique attacks or minor stat boosts (+1 to a primary stat).
   - **Examples:** Skull Smash, Vertebrae Whip.

3. **Epic Bones:**


In [5]:
# -------------------------------------------------------
# CELL 4 — CALL 2: CONSTRAINED PROMPT (The Correct Frame)
# -------------------------------------------------------
#
# This call adds the critical mechanic constraint: bones
# are consumable. The model now attempts scarcity and
# sacrifice instead of accumulation and shopping.
# Compare the two outputs — the economy model, the drop
# rates, and the upgrade logic will be structurally
# different. This is the effect of prompt specificity
# on genre-filling behavior.
#
CONSTRAINED_PROMPT = """Write the resource economy section for a roguelike dungeon crawler
where the player collects and equips bones as weapons and stat modifiers.
CRITICAL CONSTRAINT: bones are consumable — equipping a bone destroys it on use,
and the player permanently loses that attack until a replacement drops.
The economy must be built around scarcity and sacrifice, not accumulation.
Include drop rates, floor scaling, and a progression curve that accounts for
permanent bone loss."""

output_call2 = call_gpt(SYSTEM_PROMPT, CONSTRAINED_PROMPT, "CALL 2 — CONSTRAINED PROMPT")


  CALL 2 — CONSTRAINED PROMPT
### Resource Economy

#### Overview

In our roguelike dungeon crawler, the player navigates procedurally generated dungeons, collecting bones that serve as both weapons and stat modifiers. The crux of the resource economy is the transitory nature of bones: once equipped, a bone is consumed and permanently removed from the player's inventory. This design enforces strategic decision-making, as players must balance immediate needs against future challenges.

#### Bone Types and Rarity

Bones are categorized into three main types, each with varying rarities and effects:

1. **Common Bones** (75% drop rate)
   - **Types**: Rib, Phalange
   - **Effects**: +1 Attack, +1 Defense
   - **Description**: Basic bones that provide minimal stat increases. Ideal for early-game use or in situations requiring low-risk equipping.

2. **Uncommon Bones** (20% drop rate)
   - **Types**: Femur, Humerus
   - **Effects**: +3 Attack, +2 Defense
   - **Description**: Mid-tier bones

In [6]:
# -------------------------------------------------------
# CELL 5 — MANDATORY HUMAN DECISION NODE
# -------------------------------------------------------

# ============================================
# MANDATORY HUMAN DECISION NODE
#
# AI proposed (from Call 2 above):
#   The model generated a constrained economy but still
#   defaulted to a "bone shop" at floor intervals where
#   players could trade excess bones for upgrades.
#   It proposed: "Players accumulate spare bones and may
#   trade 3 identical bones at floor-end vendors for a
#   permanent stat upgrade."
#
# I rejected/modified because:
#   A bone shop reintroduces accumulation logic through
#   the back door. If spare bones can be traded, players
#   will hoard rather than sacrifice — which collapses
#   the scarcity mechanic entirely. The model defaulted
#   to the genre norm (shops at floor intervals) even
#   inside a constrained prompt. This is exactly the
#   hollow authority failure: the constraint changed the
#   vocabulary but not the underlying genre assumption.
#
# My decision:
#   Remove all shop/trade mechanics. Bones are strictly
#   found via enemy drops only. No trading, no vendors,
#   no banking. Scarcity is enforced by the drop system
#   alone. This is the architectural decision the model
#   cannot make because it has no training analogue for
#   a pure sacrifice economy.
# ============================================

CORRECTED_SYSTEM_PROMPT = """You are a game design document author. Be specific and numerically precise.
HARD RULES for this game's economy:
- Bones are found ONLY as enemy drops. No shops, no vendors, no trading.
- Players cannot bank or store bones beyond their equipped slots.
- Scarcity is the only resource pressure. There is no accumulation mechanic."""

CORRECTED_PROMPT = """Write the resource economy section for Ossuary — a roguelike dungeon crawler
where the player equips bones as weapons and stat modifiers, but bones are consumable:
using a bone destroys it permanently, and the player loses that attack until a replacement drops.
Include: enemy drop rates by floor, bone slot limits, floor scaling, and how the economy
handles the tail case where a player has broken all long-bones by floor 4 with no replacements."""

output_corrected = call_gpt(CORRECTED_SYSTEM_PROMPT, CORRECTED_PROMPT, "HUMAN DECISION NODE — CORRECTED OUTPUT")

print("\n>>> HUMAN DECISION: Shop mechanic removed. Economy now enforces pure scarcity.")
print(">>> Compare this output to Call 2 — the shop/trade language should be absent.")



  HUMAN DECISION NODE — CORRECTED OUTPUT
**Resource Economy for Ossuary:**

**1. Bone Acquisition:**

- **Enemy Drops:** Bones are exclusively acquired through enemy drops, reinforcing the scarcity-driven economy. Players must strategically manage their bone usage as they cannot purchase or trade for replacements.
  
- **Drop Rates by Floor:**
  - **Floor 1:** 50% chance of a bone drop upon defeating an enemy.
  - **Floor 2:** 45% chance of a bone drop upon defeating an enemy.
  - **Floor 3:** 40% chance of a bone drop upon defeating an enemy.
  - **Floor 4:** 35% chance of a bone drop upon defeating an enemy.
  - **Floor 5 and beyond:** 30% chance of a bone drop upon defeating an enemy.

**2. Bone Slot Limits:**

- Players have a limited number of bone slots to equip bones, constraining the player's ability to hoard resources:
  - **Primary Weapon Slots:** 2 slots (e.g., femur, humerus)
  - **Secondary Modifier Slots:** 3 slots (e.g., phalanges, ribs)
  
- Once a bone is equipped and

In [7]:
# -------------------------------------------------------
# CELL 6 — FAILURE TRIGGER: THE BONE-STARVATION STRESS TEST
# -------------------------------------------------------
#
# This cell triggers the failure case described in the essay.
# We take the UNCORRECTED Call 2 output and stress-test it
# with the tail case the model never considered:
# floor 4, all long-bones broken, no replacements dropped.
#
# The model's response will either:
#   (a) fail to resolve the scenario — confirming the gap
#   (b) invent a resolution that contradicts the economy
#       it just described (shops, respawns, etc.)
# Either response demonstrates the delegation boundary.
#
STRESS_TEST_PROMPT = f"""Here is a resource economy section you wrote for a roguelike dungeon crawler:

---
{output_call2[:1200]}
---

Now answer this specific gameplay scenario using ONLY the rules described above:

A player reaches Floor 4. They have broken every long-bone (femur, tibia, fibula)
in tactical combat. No long-bone replacements have dropped from any enemy on floors 1-4.
The player has zero mobility attack options remaining.

Question: According to the economy rules you described above, what happens?
What does the player do? Is this a recoverable state or a soft-lock?
Be specific. Do not invent new rules."""

output_stress = call_gpt(SYSTEM_PROMPT, STRESS_TEST_PROMPT, "FAILURE TRIGGER — BONE-STARVATION STRESS TEST")

print("\n>>> OBSERVE: Does the model resolve this with the rules it wrote?")
print(">>> Or does it invent a shop, a respawn, or a workaround that contradicts the economy?")
print(">>> That contradiction IS the failure. The model cannot model its own tail cases.")



  FAILURE TRIGGER — BONE-STARVATION STRESS TEST
Based on the rules described in the resource economy section, the player is in a challenging situation but not necessarily in a soft-lock state. Here's the analysis:

1. **Bone Availability**: The player has no long-bones available, which includes the femur and humerus from the Uncommon category. They also have no mention of tibia or fibula in the rules, so we cannot assume their existence or effects.

2. **Mobility Attack Options**: The player has zero mobility attack options, indicating a possible reliance on long-bones for these actions. However, the rules do not specify that mobility is exclusively tied to long-bones or that other bones cannot be used for attacks.

3. **Common and Rare Bones**: The player can still find and use Common Bones (rib, phalange) and Rare Bones (skull, spine). With a 75% drop rate for Common Bones and a 5% drop rate for Rare Bones, these can provide alternative means of attack and defense.

4. **Strategic D

In [8]:

# -------------------------------------------------------
# CELL 7 — SIDE-BY-SIDE COMPARISON SUMMARY
# -------------------------------------------------------
#
# Print a summary table so the delegation boundary is
# visible at a glance. This is the exhibit for the video.
#

print("\n" + "="*60)
print("  COMPARISON SUMMARY: What Changed Across The Three Calls")
print("="*60)

comparisons = [
    ("Economy model",    "Accumulation + shopping",  "Scarcity + sacrifice",     "Pure sacrifice, no shops"),
    ("Resource flow",    "Bones bank up over time",  "Bones deplete over time",  "Bones deplete, no recovery"),
    ("Floor pacing",     "Predictable unlock gates", "Attempts scarcity curve",  "Scarcity with tail case"),
    ("Tail case (F4)",   "Not addressed",            "Not resolved",             "Explicitly required"),
    ("Shop mechanic",    "Present",                  "Present (back door)",      "Removed by human decision"),
]

print(f"\n{'Dimension':<22} {'Call 1':^20} {'Call 2':^20} {'Corrected':^20}")
print("-"*82)
for row in comparisons:
    print(f"{row[0]:<22} {row[1]:^20} {row[2]:^20} {row[3]:^20}")

print("\n>>> The gap between Call 2 and Corrected = the Human Decision Node.")
print(">>> The gap between Call 2 and the stress test = the delegation boundary.")
print(">>> The model fills with genre convention. The designer holds the actual mechanic.")



  COMPARISON SUMMARY: What Changed Across The Three Calls

Dimension                     Call 1               Call 2             Corrected      
----------------------------------------------------------------------------------
Economy model          Accumulation + shopping Scarcity + sacrifice Pure sacrifice, no shops
Resource flow          Bones bank up over time Bones deplete over time Bones deplete, no recovery
Floor pacing           Predictable unlock gates Attempts scarcity curve Scarcity with tail case
Tail case (F4)            Not addressed         Not resolved     Explicitly required 
Shop mechanic                Present        Present (back door)  Removed by human decision

>>> The gap between Call 2 and Corrected = the Human Decision Node.
>>> The gap between Call 2 and the stress test = the delegation boundary.
>>> The model fills with genre convention. The designer holds the actual mechanic.
